STEP 0: Bring Groq's llm

In [7]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

STEP 1: Extract Text from PDF

In [8]:
# extracts text from pdf and make into langchain Documents (one page -> one Document)
# pypdf itself creates the metadata for each Document

from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader("../docs/mospi_expert_review.pdf")

docs = loader.load()

STEP 2: Convert the documents into chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

STEP 3: Create embedding model

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings 

embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2") 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9647.25it/s]


STEP 4: Create and Store embeddings in local vector db

In [11]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents( # langchain creates embeddings for those chunks and stores it in chromadb
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../vector_db" # embeddings are saved locally on this directory
)

STEP 5: Reuse the vector database, by skipping step 2 and 3

In [12]:
vector_store_persistant = Chroma(
    persist_directory='../vector_db',
    embedding_function=embedding_model # to make sure the prompt sent to chromadb also to use the same embedding model
)

STEP 5: Retrive top 3 chunks

In [ ]:
context = vector_store_persistant.similarity_search("what is meant by CPI ?") 

[Document(id='36e0a8f6-74e0-4763-b22b-84fc4fd3db92', metadata={'moddate': '2026-01-29T12:36:04+05:30', 'page': 6, 'creator': 'PyPDF', 'creationdate': '2026-01-29T12:34:54+05:30', 'producer': 'iLovePDF', 'page_label': '7', 'source': '../docs/mospi_expert_review.pdf', 'total_pages': 258}, page_content='P a g e  | 1 \n \n \nChapter 1 \nIntroduction and Process adopted for Base Revision \n \nThe Consumer Price Index (CPI) is a measure of movement of prices in the items \nconsumed by the Households. The retail inflation based on CPI which provides year on \nyear changes in CPI is a key macro -economic indicator for measuring the health of any \neconomy. Across the world, CPI is considered a very well evolved indicator in terms of \nits concepts, definitions, methodology and usage.'), Document(id='f6116d85-f7ce-40af-b7fa-9f30e1467b68', metadata={'page_label': '69', 'creationdate': '2026-01-29T12:34:54+05:30', 'page': 68, 'producer': 'iLovePDF', 'moddate': '2026-01-29T12:36:04+05:30', 'source

STEP 6: Provide context to llm to answer the user query

In [14]:
llm.invoke(f"What is Computer Science? You can answer using following context: {context}")

AIMessage(content='I’m sorry, but the documents you provided are all about the Consumer Price Index (CPI) and related economic topics. They don’t contain any information about computer science, so I can’t pull a definition or explanation of computer science from that material.\n\nIf you’d like a general overview of what computer science is, I can give you one based on common knowledge. Just let me know!', additional_kwargs={'reasoning_content': 'The user asks: "What is Computer Science? You can answer using following context: [Document...]". The provided documents are about Consumer Price Index, not computer science. The user wants an answer about computer science, but the context is about CPI. There\'s no relevant content. According to policy, we should not hallucinate. We can say that the provided documents don\'t contain information about computer science. We can offer a general answer, but we must not hallucinate from the context. We can say: The documents provided are about CPI, n